In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('../data/train_processed.csv')
df.head()

,file_1,file_2,real_text_id
0,The VIRSA (Visible Infrared Survey Telescope A...,The China relay network has released a signifi...,1
1,China\nThe goal of this project involves achie...,The project aims to achieve an accuracy level ...,2
2,Scientists can learn about how galaxies form a...,Dinosaur eggshells offer clues about what dino...,1
3,China\nThe study suggests that multiple star s...,The importance for understanding how stars evo...,2
4,Dinosaur Rex was excited about his new toy set...,Analyzing how fast stars rotate within a galax...,2


In [6]:
df.shape

(95, 3)

In [3]:
df_swap = pd.DataFrame()
df_swap['file_1'], df_swap['file_2'] = df['file_2'], df['file_1']
df_swap['real_text_id'] = 2 - df['real_text_id']

df_swap.head()

,file_1,file_2,real_text_id
0,The China relay network has released a signifi...,The VIRSA (Visible Infrared Survey Telescope A...,1
1,The project aims to achieve an accuracy level ...,China\nThe goal of this project involves achie...,0
2,Dinosaur eggshells offer clues about what dino...,Scientists can learn about how galaxies form a...,1
3,The importance for understanding how stars evo...,China\nThe study suggests that multiple star s...,0
4,Analyzing how fast stars rotate within a galax...,Dinosaur Rex was excited about his new toy set...,0


In [4]:
final = pd.concat([df, df_swap], axis=0)
final.head()

,file_1,file_2,real_text_id
0,The VIRSA (Visible Infrared Survey Telescope A...,The China relay network has released a signifi...,1
1,China\nThe goal of this project involves achie...,The project aims to achieve an accuracy level ...,2
2,Scientists can learn about how galaxies form a...,Dinosaur eggshells offer clues about what dino...,1
3,China\nThe study suggests that multiple star s...,The importance for understanding how stars evo...,2
4,Dinosaur Rex was excited about his new toy set...,Analyzing how fast stars rotate within a galax...,2


In [5]:
final.shape

(190, 3)

In [7]:
final.rename(columns={'file_1': 'text_1', 'file_2': 'text_2', 'real_text_id': 'label'}, inplace=True)

In [10]:
from nltk import word_tokenize, sent_tokenize

In [16]:
import nltk
nltk.download('punkt_tab')

# redefine the feature extractor to return the features dict
def generate_features(text):
    features = {}
    features['num_chars'] = len(text)
    features['num_words'] = len(word_tokenize(text))
    features['num_sentences'] = len(sent_tokenize(text))
    features['avg_word_length'] = features['num_chars'] / features['num_words'] if features['num_words'] > 0 else 0
    features['avg_sentence_length'] = features['num_words'] / features['num_sentences'] if features['num_sentences'] > 0 else 0
    return features

# recompute features now that the function returns properly (overwrites previous variables)
features_1 = final['text_1'].fillna('').astype(str).apply(generate_features).apply(pd.Series).add_suffix('_1')
features_2 = final['text_2'].fillna('').astype(str).apply(generate_features).apply(pd.Series).add_suffix('_2')
# join features back to the dataframe
final = pd.concat([final, features_1, features_2], axis=1)

final.head()

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


,text_1,text_2,label,num_chars_1,num_words_1,num_sentences_1,avg_word_length_1,avg_sentence_length_1,num_chars_2,num_words_2,num_sentences_2,avg_word_length_2,avg_sentence_length_2
0,The VIRSA (Visible Infrared Survey Telescope A...,The China relay network has released a signifi...,1,2196.0,322.0,9.0,6.819876,35.777778,2018.0,323.0,10.0,6.247678,32.300000
1,China\nThe goal of this project involves achie...,The project aims to achieve an accuracy level ...,2,3124.0,497.0,8.0,6.285714,62.125000,936.0,153.0,4.0,6.117647,38.250000
2,Scientists can learn about how galaxies form a...,Dinosaur eggshells offer clues about what dino...,1,1139.0,171.0,3.0,6.660819,57.000000,801.0,133.0,3.0,6.022556,44.333333
3,China\nThe study suggests that multiple star s...,The importance for understanding how stars evo...,2,1774.0,313.0,8.0,5.667732,39.125000,1869.0,271.0,6.0,6.896679,45.166667
4,Dinosaur Rex was excited about his new toy set...,Analyzing how fast stars rotate within a galax...,2,195.0,38.0,3.0,5.131579,12.666667,871.0,138.0,4.0,6.311594,34.500000


In [17]:
final.to_csv('../data/train_final.csv', index=False)

In [18]:
test = pd.read_csv('../data/test_processed.csv')
test.head()

,sample_id,file_1.txt_context,file_2.txt_context
0,article_0000,"""Music"" Music music music Music music Music mu...",Since its launch on Paranal observatory's Very...
1,article_0001,underground exploration on SN's birth has prov...,SN 1987A provides valuable insights as newer o...
2,article_0002,This research aimed to understand how star sha...,ChromeDriver music player\nThis study focused ...
3,article_0003,Using OmegaCAM's wide field capabilities spann...,"greek translation :\nvazhi (megaCAM), territor..."
4,article_0004,AssemblyCulture AssemblyCulture AssemblyCultur...,XClass is software tool that helps astronomers...


In [19]:
test.drop(columns=['sample_id'], inplace=True)

In [20]:
test.rename(columns={'file_1.txt_context': 'text_1', 'file_2.txt_context': 'text_2'}, inplace=True)

In [21]:
features_1_test = test['text_1'].fillna('').astype(str).apply(generate_features).apply(pd.Series).add_suffix('_1')
features_2_test = test['text_2'].fillna('').astype(str).apply(generate_features).apply(pd.Series).add_suffix('_2')
# join features back to the dataframe
test = pd.concat([test, features_1_test, features_2_test], axis=1)

test.head()

,text_1,text_2,num_chars_1,num_words_1,num_sentences_1,avg_word_length_1,avg_sentence_length_1,num_chars_2,num_words_2,num_sentences_2,avg_word_length_2,avg_sentence_length_2
0,"""Music"" Music music music Music music Music mu...",Since its launch on Paranal observatory's Very...,1710.0,277.0,8.0,6.173285,34.625000,1249.0,185.0,4.0,6.751351,46.250000
1,underground exploration on SN's birth has prov...,SN 1987A provides valuable insights as newer o...,1168.0,188.0,5.0,6.212766,37.600000,1516.0,233.0,6.0,6.506438,38.833333
2,This research aimed to understand how star sha...,ChromeDriver music player\nThis study focused ...,752.0,121.0,3.0,6.214876,40.333333,1436.0,239.0,6.0,6.008368,39.833333
3,Using OmegaCAM's wide field capabilities spann...,"greek translation :\nvazhi (megaCAM), territor...",1223.0,181.0,6.0,6.756906,30.166667,201.0,36.0,4.0,5.583333,9.000000
4,AssemblyCulture AssemblyCulture AssemblyCultur...,XClass is software tool that helps astronomers...,1271.0,220.0,7.0,5.777273,31.428571,1748.0,253.0,8.0,6.909091,31.625000


In [22]:
test.to_csv('../data/test_final.csv', index=False)